# General Relativity Module

This module provides the foundational tools and numerical infrastructure needed to work with General Relativity (GR) in a modern, performant, and extensible way. It is designed to support both symbolic and numerical computations related to curved spacetimes, tensorial quantities, and dynamical evolutions in a 4D or 3+1 formulation.

---

## Objectives

- Represent tensors, metrics, and connections in arbitrary coordinate systems.
- Support symbolic parsing of Einstein field equations and related expressions.
- Provide efficient numerical backends for computing curvature (Christoffel symbols, Riemann, Ricci, Einstein tensors).
- Support BSSN formalism and numerical relativity applications.

---


## Key Features

- High-performance tensor algebra using SIMD and OpenMP.
- Built-in support for Kerr, Schwarzschild, Minkowski, and user-defined metrics.
- Futur integration with numerical evolution schemes (RK4, flux solvers).

---

## When to Use This Module

This module is intended for developers, researchers, and students working on:

- Black hole simulations
- Cosmological modeling
- Numerical relativity
- High-performance scientific computing


### Schwarzschild Metric and Its Inverse

In this example, we demonstrate how to define and use a custom metric in Morpheus.  
(The Schwarzschild metric is already available via the built-in `morpheus_RG::Metric<double> metric("Schwarzschild", 1.0, 0.0);` constructor, where the two arguments are mass and spin.)

In general relativity, the Schwarzschild metric describes the curvature of spacetime around a static, spherically symmetric mass.  
Its line element is given by:

$$
ds^2 = -\left(1 - \frac{2M}{r} \right) dt^2 + \left(1 - \frac{2M}{r} \right)^{-1} dr^2 + r^2 d\theta^2 + r^2 \sin^2 \theta \, d\phi^2
$$

This corresponds to a diagonal $4 \times 4$ metric tensor $g_{\mu\nu}$ in spherical coordinates $(t, r, \theta, \phi)$:

$$
g_{\mu\nu} = \begin{pmatrix}
- f(r) & 0 & 0 & 0 \\
0 & f(r)^{-1} & 0 & 0 \\
0 & 0 & r^2 & 0 \\
0 & 0 & 0 & r^2 \sin^2 \theta
\end{pmatrix}, \quad \text{with } f(r) = 1 - \frac{2M}{r}
$$

Although this matrix is diagonal and its inverse is analytically trivial, we compute the inverse numerically using a general-purpose matrix inversion routine.  
This serves both as a validation of the numerical backend and a practical example for working with symbolic or nontrivial metrics.

This example is a foundational step toward computing Christoffel symbols, the Ricci tensor, and full curvature tensors in numerical relativity simulations.

In [11]:
import sys
import time 
import os
sys.path.append(os.path.abspath("../pybuild"))
from morpheus import *
from morpheus import Matrix, morph
import math

def schwarzschild_metric(r, theta, M=1.0):
    g = Matrix(4, 4)
    f = 1.0 - 2.0 * M / r

    g[0, 0] = -f
    g[1, 1] = 1.0 / f
    g[2, 2] = r**2
    g[3, 3] = r**2 * math.sin(theta)**2

    return g

r = 10.0
theta = math.pi / 4
g = schwarzschild_metric(r, theta)

g_inv = morph.inverse_mat(g)

print("Metric g_mu_nu at (r, θ):")
print(g)
print("Inverse metric g^mu_nu:")
print(g_inv)


Metric g_mu_nu at (r, θ):
[
  [-0.8, 0, 0, 0],
  [0, 1.25, 0, 0],
  [0, 0, 100, 0],
  [0, 0, 0, 50]
]
Inverse metric g^mu_nu:
[
  [-1.25, 0, 0, 0],
  [0, 0.8, 0, 0],
  [0, 0, 0.01, 0],
  [0, 0, 0, 0.02]
]


### Kerr Geometry: Computing the Riemann Curvature Tensor

In this example, we compute the **Riemann curvature tensor** $R^\lambda_{\ \mu\nu\rho}$ at a specific spacetime point in the Kerr metric, a solution to Einstein's field equations representing a rotating black hole.

We evaluate at the following point in Boyer–Lindquist coordinates:

$$
X^\mu = (t, r, \theta, \phi) = \left(0,\ 10,\ \frac{\pi}{2},\ 0\right)
$$

with Kerr parameters:
- $M = 1.0$ (mass of the black hole)
- $a = 0.8$ (angular momentum per unit mass)

---

#### 1. Metric Tensor

We construct the Kerr metric $g_{\mu\nu}(X)$ at this point with:

```python
g = morph.Metric("kerr", 1.0, 0.8)(X)
```

This returns a $4 \times 4$ symmetric matrix representing the spacetime geometry.  
We compute its inverse $g^{\mu\nu}$ via:

```python
g_inv = morph.inv_mat_tensor(g)
```

---

#### 2. Christoffel Symbols

The Christoffel symbols $\Gamma^\lambda_{\mu\nu}$ describe how vectors are transported in curved space and are computed as:

```python
Gamma = morph.compute_christoffel(X, g, g_inv, "kerr", 1.0, 0.8)
```

Their definition is:

$$
\Gamma^\lambda_{\mu\nu} = \frac{1}{2} g^{\lambda\sigma} \left( \partial_\mu g_{\nu\sigma} + \partial_\nu g_{\mu\sigma} - \partial_\sigma g_{\mu\nu} \right)
$$

Finite difference approximations are used for the partial derivatives.

---

#### 3. Riemann Tensor

The Riemann tensor is computed from the Christoffel symbols using:

```python
R = morph.compute_riemann_tensor(X, "kerr", 1.0, 0.8)
```

The formula is:

$$
R^\lambda_{\ \mu\nu\rho} = \partial_\nu \Gamma^\lambda_{\mu\rho}
- \partial_\rho \Gamma^\lambda_{\mu\nu}
+ \Gamma^\lambda_{\nu\sigma} \Gamma^\sigma_{\mu\rho}
- \Gamma^\lambda_{\rho\sigma} \Gamma^\sigma_{\mu\nu}
$$

It encodes the tidal and curvature effects of spacetime.

We print the components sliced by upper index $\lambda$:

```python
morph.Riemann.print_componentwise(R)
```

---

### Summary

This pipeline performs a **numerical differential geometry evaluation** in general relativity:
- Builds the Kerr metric analytically
- Computes the Christoffel symbols $\Gamma^\lambda_{\mu\nu}$ numerically
- Computes the Riemann tensor $R^\lambda_{\ \mu\nu\rho}$
- Displays the full curvature structure at the point $X^\mu$

This reveals how spacetime is curved near a rotating black hole, illustrating gravitational tidal forces and frame-dragging effects.


In [ ]:
from morpheus import Vectord, morph

X = Vectord([0.0, 10.0, 3.14159 / 2, 0.0])
g = morph.Metric("kerr", 1.0, 0.935)(X)
g_inv = morph.inv_mat_tensor(g)
Gamma = morph.compute_christoffel(X, g, g_inv, "kerr", 1.0, 0.935)
Gamma.print()
Riemann = morph.compute_riemann_tensor(X, "kerr", 1.0, 0.935)
Ricci = morph.contract_riemann_to_ricci(Riemann, g_inv)
morph.Riemann.print_componentwise(Riemann)
morph.print_ricci_tensor(Ricci)
Rscalar = morph.print_ricci_scalar(Ricci, g_inv)



1.2203315289830566e-19
Γ^0_{μν} :
    0.000000     0.012473    -0.000000     0.000000 
    0.012473     0.000000     0.000000    -0.034785 
   -0.000000     0.000000     0.000000     0.000000 
    0.000000    -0.034785     0.000000     0.000000 

Γ^1_{μν} :
    0.008087     0.000000     0.000000    -0.007562 
    0.000000    -0.011284    -0.000000     0.000000 
    0.000000    -0.000000    -8.087422     0.000000 
   -0.007562     0.000000     0.000000    -8.080352 

Γ^2_{μν} :
   -0.000000     0.000000     0.000000     0.000000 
    0.000000     0.000000     0.100000     0.000000 
    0.000000     0.100000    -0.000000     0.000000 
    0.000000     0.000000     0.000000    -0.000001 

Γ^3_{μν} :
    0.000000     0.000116    -0.000000     0.000000 
    0.000116     0.000000     0.000000     0.098811 
   -0.000000     0.000000     0.000000     0.000001 
    0.000000     0.098811     0.000001     0.000000 


Riemann tensor components (sliced by upper index λ):
R^0_{μνρ} :
      0.000000 